# Breast — SLC transporter factors

Copy of `signature_analysis_breast.ipynb` that views the **SLC transporter** factor
block instead of the cancer neighborhood factors. SLC factors are **cell-autonomous**
(each T cell's own SLC-gene RNA, added like a TF — no spatial diffusion), so the
`custom`-group factors shown in every `view()` ratio plot below are SLC transporters.
The nbhd-only spatial-panel / interaction-heatmap sections are intentionally omitted.

Run `signature_run.py --config breast --factor-group slc` first to populate the
`{name}/fg_slc` cache this notebook reads.

In [ ]:
import sys
import json
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from metab_processing.LinearRegression import signature_helpers as sh, signature_batch as sb
from metab_processing.LinearRegression.signature_configs import CONFIGS
from dataclasses import replace

# SLC variant: cell-autonomous SLC transporter factors instead of the cancer
# neighborhood factors. `factor_group='slc'` routes reads to the {name}/fg_slc cache
# (populated by `signature_run.py --config breast --factor-group slc`).
cfg = replace(CONFIGS['breast'], factor_group='slc')

# Which ratio plots `view()` renders (keys of sh.plot_res).
PLOTS = ('ratio', 'variance_ratio', 'meansq_variance_ratio')
# ---- Which cached fit to READ (must match a computed params subdir) ----
# Default = this config's fit (l1 @ penalty 0.005). To read a different fit, override, e.g.:
#   READ = dict(method='ridge', penalty=800)
READ = dict(method=cfg.method, penalty=cfg.penalty, l1_ratio=cfg.l1_ratio)

In [ ]:
membership = json.load(open(Path(cfg.cache_dir) / '_membership.json'))
print(f'{cfg.name}: loaded membership for {len(membership)} gene set(s)')

## SLC factors in this run

`_SLC_FACTORS` lists the SLC genes added as cell-autonomous predictors. A target
signature gene that is itself an SLC factor is dropped from that target (shown as
`slc_removed` above) to avoid a self-term in both X and y.

In [ ]:
slc = membership.get('_SLC_FACTORS', {})
print(f"SLC factors ({slc.get('total', 0)}): {slc.get('present', [])}")

## breast: target signatures -- membership, missing, present, factor-removed, remaining

In [ ]:
print(f'===== {cfg.name} target signatures: membership + factor-removal =====')
for name in list(cfg.target_signatures) + list(cfg.neg_controls) + list(cfg.manual_sets):
    m = membership.get(name, {})
    print(f'\n{name}')
    print(f"  present ({len(m.get('present', []))}): {m.get('present', [])}")
    print(f"  missing ({len(m.get('missing', []))}): {m.get('missing', [])}")
    print(f"  total: {m.get('total', 0)}")
    print(f"  factor_removed ({len(m.get('factor_removed', []))}): {m.get('factor_removed', [])}")
    print(f"  remaining ({len(m.get('remaining', []))}): {m.get('remaining', [])}")
    print(f"  slc_removed ({len(m.get('slc_removed', []))}): {m.get('slc_removed', [])}")

## View helper -- `view(sig)` renders every cached run-spec for one signature

In [ ]:
TARGETS = sb.all_targets(cfg)   # targets + neg controls + manual sets (+ colon single gene)
specs = sb.runspecs(cfg)


def view(sig):
    for spec in specs:
        res = sb.load_res(cfg, sig, spec['label'], **READ)
        if res is None:
            print('not yet computed:', sig, spec['label'])
            continue
        sh.plot_res(res, highlight_genes=membership.get(sig, {}).get('present', []),
                    title=f"{sig} | {spec['label']}", plots=PLOTS)

## Metabolic targets (predicted like the T-cell signatures)

In [ ]:
view('HALLMARK_HYPOXIA')

In [ ]:
view('HALLMARK_GLYCOLYSIS')

In [ ]:
view('HALLMARK_OXIDATIVE_PHOSPHORYLATION')

In [ ]:
view('HALLMARK_FATTY_ACID_METABOLISM')

In [ ]:
view('HALLMARK_MTORC1_SIGNALING')

## Subsample stats from cached raw betas -- call `stats('SIGNATURE_NAME')`

In [ ]:
# 95% CI / selection-frequency / stability, computed directly from the cached raw subsample
# betas. (CI at n_sub resamples is coarse; under L1 betas pile at 0 so selection_freq + a
# CI excluding 0 are the honest signals, p_stability is a conservative heuristic.)
from IPython.display import display

def stats(sig):
    for spec in specs:
        betas = sb.load_betas(cfg, sig, spec['label'], **READ)
        if betas is None:
            print('not yet computed:', sig, spec['label']); continue
        print(f"\n=== {sig} | {spec['label']} ===")
        display(sh.subsample_stats(betas).sort_values('selection_freq', ascending=False).head(25))

## T-cell TARGET signatures

### HALLMARK_IL2_STAT5_SIGNALING

In [ ]:
view('HALLMARK_IL2_STAT5_SIGNALING')

### KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY

In [ ]:
view('KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY')

### GOBP_T_CELL_MEDIATED_CYTOTOXICITY

In [ ]:
view('GOBP_T_CELL_MEDIATED_CYTOTOXICITY')

### KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY

In [ ]:
view('KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY')

### HALLMARK_TNFA_SIGNALING_VIA_NFKB

In [ ]:
view('HALLMARK_TNFA_SIGNALING_VIA_NFKB')

### HALLMARK_INTERFERON_GAMMA_RESPONSE

In [ ]:
view('HALLMARK_INTERFERON_GAMMA_RESPONSE')

### HALLMARK_INTERFERON_ALPHA_RESPONSE

In [ ]:
view('HALLMARK_INTERFERON_ALPHA_RESPONSE')

### GOBP_CELL_KILLING

In [ ]:
view('GOBP_CELL_KILLING')

### GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY

In [ ]:
view('GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY')

### GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY

In [ ]:
view('GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY')

## Negative controls

### HALLMARK_PANCREAS_BETA_CELLS

In [ ]:
view('HALLMARK_PANCREAS_BETA_CELLS')

### HALLMARK_SPERMATOGENESIS

In [ ]:
view('HALLMARK_SPERMATOGENESIS')

## MANUAL_ISLET_NEG_CONTROL (manual negative control)

In [ ]:
view('MANUAL_ISLET_NEG_CONTROL')